In [1]:
import os
os.getcwd()

'/data02/liaoyy/GST_ESM3/molecular_docking'

In [ ]:
# !conda install -c conda-forge openbabel
# !pip install prody
# !conda install -c conda-forge pdbfixer openmm

In [24]:
!wget https://files.rcsb.org/download/1Y6E.cif
!wget https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/124886/SDF -O GSH.sdf
!cp -r ../output/identity_filtered_esmfold .

--2025-11-16 13:06:24--  https://files.rcsb.org/download/1Y6E.cif
Resolving files.rcsb.org (files.rcsb.org)... 3.166.135.84, 3.166.135.67, 3.166.135.66
Connecting to files.rcsb.org (files.rcsb.org)|3.166.135.84|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [chemical/x-cif]
Saving to: ‘1Y6E.cif’

1Y6E.cif                [        <=>         ] 384.31K   191KB/s    in 2.0s    

2025-11-16 13:06:28 (191 KB/s) - ‘1Y6E.cif’ saved [393532]

--2025-11-16 13:06:29--  https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/124886/SDF
Resolving pubchem.ncbi.nlm.nih.gov (pubchem.ncbi.nlm.nih.gov)... 130.14.29.110, 2607:f220:41e:4290::110
Connecting to pubchem.ncbi.nlm.nih.gov (pubchem.ncbi.nlm.nih.gov)|130.14.29.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [chemical/x-mdl-sdfile]
Saving to: ‘GSH.sdf’

GSH.sdf                 [ <=>                ]   5.55K  --.-KB/s    in 0s      

2025-11-16 13:06:30 (449 MB/s) - ‘

0. imports & paths

In [13]:
import os
import glob
import subprocess
import numpy as np
import prody
from prody import (parseCIF, writePDB, parsePDB, calcTransformation, 
                   applyTransformation, calcCenter, LOGGER, matchAlign)

from pdbfixer import PDBFixer
from openmm.app import *
from openmm import unit


WORK_DIR = "/data02/liaoyy/GST_ESM3/molecular_docking"
os.chdir(WORK_DIR)

REFERENCE_CIF = '1Y6E.cif'
LIGAND_SDF = 'GSH.sdf'
LIGAND_NAME = 'GSH'

ESMFOLD_DIR = 'identity_filtered_esmfold'

PREPROCESSED_DIR = 'preprocessed'
SUPERIMPOSED_DIR = 'superimposed_identity_filtered_esmfold'
PDBQT_RECEPTOR_DIR = 'pdbqt_receptors'
VINA_OUTPUT_DIR = 'vina_docking_results'
FINAL_FASTA_FILE = 'top_40_docking_hits.fasta'

os.makedirs(PREPROCESSED_DIR, exist_ok=True)
os.makedirs(SUPERIMPOSED_DIR, exist_ok=True)
os.makedirs(PDBQT_RECEPTOR_DIR, exist_ok=True)
os.makedirs(VINA_OUTPUT_DIR, exist_ok=True)

BOX_SIZE = [20.0, 20.0, 20.0]
VINA_EXHAUSTIVENESS = 32 


1. preprocessing

In [3]:
try:
    structure = parseCIF(REFERENCE_CIF)
except Exception as e:
    raise

reference_chain_A = structure.select('protein and chain A')

REFERENCE_PDB_CLEAN = os.path.join(PREPROCESSED_DIR, '1Y6E_A_clean.pdb')
writePDB(REFERENCE_PDB_CLEAN, reference_chain_A)
    
print(f"Successfully preprocessed reference structure: {REFERENCE_PDB_CLEAN}")
print(f"Maintain {reference_chain_A.numResidues()} residues after preprocessing.")

Successfully preprocessed reference structure: preprocessed/1Y6E_A_clean.pdb
Maintain 216 residues after preprocessing.


2. superimposition

In [27]:
reference_structure = parsePDB(REFERENCE_PDB_CLEAN)

pdb_files = glob.glob(os.path.join(ESMFOLD_DIR, 'generated_sequence_*.pdb'))
print(f"Found {len(pdb_files)} structrues")

superimposed_count = 0
for pdb_file in pdb_files:
    try:
        basename = os.path.basename(pdb_file)
        output_name = f"superimposed_{basename}"
        output_path = os.path.join(SUPERIMPOSED_DIR, output_name)
        
        mobile_structure = parsePDB(pdb_file)
        
        result_tuple = matchAlign(mobile_structure, reference_structure,
                                  seqid=1,     
                                  overlap=10,
                                  pwalign=True)

        superposed_mobile = result_tuple[0]
        
        match_identity = result_tuple[3]
        match_overlap = result_tuple[4]
        print(f"  - Success: {basename} superimposed (Identity: {match_identity:.2f}%, Overlap: {match_overlap:.2f}%)")

        writePDB(output_path, superposed_mobile, conect=False)
        superimposed_count += 1

    except Exception as e:
        print(e)

print(f"\nSave: {superimposed_count} structures to {SUPERIMPOSED_DIR}")

Found 176 structrues
  - Success: generated_sequence_442.pdb superimposed (Identity: 32.87%, Overlap: 88.89%)
  - Success: generated_sequence_487.pdb superimposed (Identity: 30.09%, Overlap: 93.98%)
  - Success: generated_sequence_137.pdb superimposed (Identity: 36.11%, Overlap: 91.67%)
  - Success: generated_sequence_502.pdb superimposed (Identity: 31.94%, Overlap: 92.59%)
  - Success: generated_sequence_364.pdb superimposed (Identity: 29.63%, Overlap: 89.81%)
  - Success: generated_sequence_654.pdb superimposed (Identity: 31.94%, Overlap: 91.20%)
  - Success: generated_sequence_479.pdb superimposed (Identity: 28.24%, Overlap: 92.13%)
  - Success: generated_sequence_444.pdb superimposed (Identity: 33.80%, Overlap: 83.80%)
  - Success: generated_sequence_124.pdb superimposed (Identity: 31.48%, Overlap: 94.91%)
  - Success: generated_sequence_850.pdb superimposed (Identity: 33.33%, Overlap: 89.35%)
  - Success: generated_sequence_723.pdb superimposed (Identity: 40.28%, Overlap: 88.89%)


3. locate docking box

In [4]:
prompt_residues_spec = {
    5: "Y", 6: "W", 43: "K", 52: "N", 53: "L",
    65: "Q", 66: "S", 99: "D", 109: "Y"
}


res_map_3_to_1 = prody.atomic.AAMAP

reference_hv = reference_chain_A.getHierView()

residues_list = list(reference_hv.iterResidues())

atom_selection_list = []
print("validate residues:")
for index, expected_code_1 in prompt_residues_spec.items():
    residue = residues_list[index]
    
    resnum = residue.getResnum() 
    resname_3 = residue.getResname()
    resname_1 = res_map_3_to_1.get(resname_3, '?')
    
    if resname_1 == expected_code_1:
        print(f"  - index {index} -> ResNum {resnum} ('{resname_3}') - match '{expected_code_1}'")
        atom_selection_list.append(f"resnum {resnum}")
    else:
        print(f"  - warning: index {index} (ResNum {resnum}) residue {resname_3} ('{resname_1}'), "
                f"expected '{expected_code_1}'.")

selection_string = " or ".join(atom_selection_list)

print(f"\nProDy selected: '{selection_string}'")

active_site_atoms = reference_chain_A.select(selection_string)

box_center = calcCenter(active_site_atoms)

print(f"  central (x, y, z): {box_center[0]:.3f}, {box_center[1]:.3f}, {box_center[2]:.3f}")
print(f"  size (x, y, z): {BOX_SIZE[0]:.1f}, {BOX_SIZE[1]:.1f}, {BOX_SIZE[2]:.1f}")

validate residues:
  - index 5 -> ResNum 6 ('TYR') - match 'Y'
  - index 6 -> ResNum 7 ('TRP') - match 'W'
  - index 43 -> ResNum 44 ('LYS') - match 'K'
  - index 52 -> ResNum 53 ('ASN') - match 'N'
  - index 53 -> ResNum 54 ('LEU') - match 'L'
  - index 65 -> ResNum 66 ('GLN') - match 'Q'
  - index 66 -> ResNum 67 ('SER') - match 'S'
  - index 99 -> ResNum 100 ('ASP') - match 'D'
  - index 109 -> ResNum 110 ('TYR') - match 'Y'

ProDy selected: 'resnum 6 or resnum 7 or resnum 44 or resnum 53 or resnum 54 or resnum 66 or resnum 67 or resnum 100 or resnum 110'
  central (x, y, z): 18.148, 21.074, -8.745
  size (x, y, z): 20.0, 20.0, 20.0


4. prepare ligand GSH

In [29]:
!mk_prepare_ligand.py -i GSH.sdf -o GSH.pdbqt

/data02/liaoyy/anaconda3/envs/vina/lib/python3.10/site-packages/meeko/molsetup.py:1584: RuntimeWarning: RDKit molecule not labeled as 3D. This warning won't show again.
  warnings.warn(
Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


5. prepare proteins GST

In [30]:
import glob
import os
import subprocess
import re

center_str = [f"{c:.3f}" for c in box_center]
size_str = [f"{s:.1f}" for s in BOX_SIZE]

os.makedirs(PDBQT_RECEPTOR_DIR, exist_ok=True)

superimposed_pdbs = glob.glob(os.path.join(SUPERIMPOSED_DIR, 'superimposed_*.pdb'))
print(f"found {len(superimposed_pdbs)} superimposed pdb")

prepared_count = 0
for pdb_path in superimposed_pdbs:
    
    basename = os.path.basename(pdb_path).replace('.pdb', '')
    print(f"basename: {basename}")
    output_base_path = os.path.join(PDBQT_RECEPTOR_DIR, basename)

    meeko_cmd = [
        'mk_prepare_receptor.py',
        '--read_pdb', pdb_path,
        '-o', output_base_path,
        '-p', 
        '-v', 
        '--box_size'] + size_str + [
        '--box_center'] + center_str
    
    try:
        subprocess.run(meeko_cmd, check=True, text=True, capture_output=True)
        
        if os.path.exists(output_base_path + ".pdbqt") and os.path.exists(output_base_path + ".box.txt"):
            prepared_count += 1
        else:
            print(f"ERROR: {basename} cannot get output in the first try.")
            
    except subprocess.CalledProcessError as e:
        error_log = e.stderr
        
        clash_residues = re.findall(r"matched with excess inter-residue bond\(s\): (A:\d+)", error_log)
        
        if clash_residues:
            unique_residues = sorted(list(set(clash_residues)))

            residues_to_delete_str = ",".join(unique_residues)
            
            print(f"try to delete residues '{residues_to_delete_str}'")

            meeko_cmd_retry = [
                'mk_prepare_receptor.py',
                '--read_pdb', pdb_path,
                '-o', output_base_path,
                '-p', 
                '-v', 
                '--delete_residues', residues_to_delete_str,
                '--box_size'] + size_str + [
                '--box_center'] + center_str
            
            try:
                subprocess.run(meeko_cmd_retry, check=True, text=True, capture_output=True)
                
                if os.path.exists(output_base_path + ".pdbqt") and os.path.exists(output_base_path + ".box.txt"):
                    prepared_count += 1
                    print(f"SUCCESS: {basename} ")
                else:
                    print(f"ERROR: {basename} cannot get output in the second try.")

            except subprocess.CalledProcessError as e2:
                if e2.stderr: print(f"     Stderr: {e2.stderr[:500]}...") 
        
        else:
            if e.stdout: print(f"     Stdout: {e.stdout}")
            if e.stderr: print(f"     Stderr: {e.stderr[:500]}...")

print(f"\nfinish: {prepared_count} proteins prepared and saved in {PDBQT_RECEPTOR_DIR}")

found 176 superimposed pdb
basename: superimposed_generated_sequence_541
basename: superimposed_generated_sequence_137
try to delete residues 'A:189,A:25'
SUCCESS: superimposed_generated_sequence_137 
basename: superimposed_generated_sequence_390
basename: superimposed_generated_sequence_880
basename: superimposed_generated_sequence_687
try to delete residues 'A:14,A:201'
SUCCESS: superimposed_generated_sequence_687 
basename: superimposed_generated_sequence_286
basename: superimposed_generated_sequence_912
basename: superimposed_generated_sequence_784
try to delete residues 'A:147,A:90'
SUCCESS: superimposed_generated_sequence_784 
basename: superimposed_generated_sequence_57
basename: superimposed_generated_sequence_569
basename: superimposed_generated_sequence_463
try to delete residues 'A:140,A:166,A:204,A:22,A:68,A:94,A:96,A:99'
SUCCESS: superimposed_generated_sequence_463 
basename: superimposed_generated_sequence_502
basename: superimposed_generated_sequence_969
basename: superi

6. perform docking

In [11]:
receptor_pdbqts = glob.glob(os.path.join(PDBQT_RECEPTOR_DIR, 'superimposed_*.pdbqt'))
print(f"found {len(receptor_pdbqts)} PDBQT receptors")

LIGAND_PDBQT = "GSH.pdbqt"

docking_count = 0

for receptor_path in receptor_pdbqts:
    
    basename = os.path.basename(receptor_path).replace('.pdbqt', '')
    
    config_file = os.path.join(PDBQT_RECEPTOR_DIR, basename + '.box.txt')
    if not os.path.exists(config_file):
        print(f"cannot find {config_file}, skip {basename}")
        continue
        
    output_vina_pdbqt = os.path.join(VINA_OUTPUT_DIR, f"{basename}_vina_out.pdbqt")
    output_vina_log = os.path.join(VINA_OUTPUT_DIR, f"{basename}_vina_log.txt")
    
    vina_cmd = [
        'vina',
        '--receptor', receptor_path,
        '--ligand', LIGAND_PDBQT,
        '--config', config_file,
        '--exhaustiveness', str(VINA_EXHAUSTIVENESS),
        '--out', output_vina_pdbqt,
        '--log', output_vina_log
    ]
    
    print(f"docking: {basename}...")
    try:
        subprocess.run(vina_cmd, check=True, capture_output=False, text=True)
        docking_count += 1
    except subprocess.CalledProcessError as e:
        print(f"error: {basename} failed")
        print(f"command: {' '.join(e.cmd)}")
        print(f"stderr: {e.stderr}")

print(f"\nfinish: {docking_count} dockings, saved {VINA_OUTPUT_DIR}")

found 175 PDBQT receptors
docking: superimposed_generated_sequence_11...
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, Journal of Computational Chemistry 31 (2010)  #
# 455-461                                                       #
#                                                               #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see http://vina.scripps.edu for more information.      #
#################################################################

Detected 104 CPUs
Reading input ... done.
Setting up the scoring fun

7. extract top-40

In [14]:
import re

def get_best_vina_score(filepath):
    score_pattern = re.compile(r"^REMARK VINA RESULT:\s+?(-?\d+\.\d+)\s+")
    with open(filepath, 'r') as f:
        for line in f:
            match = score_pattern.match(line)
            if match:
                return float(match.group(1))
    return None

vina_results = glob.glob(os.path.join(VINA_OUTPUT_DIR, '*_vina_out.pdbqt'))
scores = {}

for result_file in vina_results:
    base_name = os.path.basename(result_file)
    original_name = base_name.replace('superimposed_', '').replace('_vina_out.pdbqt', '')
    
    score = get_best_vina_score(result_file)
    if score is not None:
        scores[original_name] = score

sorted_scores = sorted(scores.items(), key=lambda item: item[1])

print("top-40:")
for name, score in sorted_scores[:40]:
    print(f"  - {name}: {score:.3f} kcal/mol")
    
top_40_hits = sorted_scores[:40]

fasta_content = []
for name, score in top_40_hits:
    original_pdb_path = os.path.join(ESMFOLD_DIR, f"{name}.pdb")
    
    if not os.path.exists(original_pdb_path):
        print(f"cannot find original pdb {original_pdb_path}, skip {name}")
        continue
        
    try:
        prot = parsePDB(original_pdb_path)
        sequence = prot.select('protein and name CA').getSequence()
        
        fasta_header = f">{name} | Vina_Score={score:.3f}"
        fasta_content.append(fasta_header)
        fasta_content.append(sequence)
            
    except Exception as e:
        print(f"error: cannot extract {original_pdb_path}: {e}")
        
with open(FINAL_FASTA_FILE, 'w') as f:
    f.write("\n".join(fasta_content))
    
print(f"\nfinish, Top {len(top_40_hits)} sequences saved in {FINAL_FASTA_FILE}")

top-40:
  - generated_sequence_169: -7.500 kcal/mol
  - generated_sequence_994: -7.400 kcal/mol
  - generated_sequence_490: -7.300 kcal/mol
  - generated_sequence_463: -7.200 kcal/mol
  - generated_sequence_687: -7.100 kcal/mol
  - generated_sequence_694: -6.900 kcal/mol
  - generated_sequence_124: -6.900 kcal/mol
  - generated_sequence_269: -6.900 kcal/mol
  - generated_sequence_502: -6.800 kcal/mol
  - generated_sequence_772: -6.800 kcal/mol
  - generated_sequence_479: -6.700 kcal/mol
  - generated_sequence_758: -6.700 kcal/mol
  - generated_sequence_345: -6.700 kcal/mol
  - generated_sequence_234: -6.600 kcal/mol
  - generated_sequence_93: -6.600 kcal/mol
  - generated_sequence_729: -6.500 kcal/mol
  - generated_sequence_211: -6.500 kcal/mol
  - generated_sequence_13: -6.500 kcal/mol
  - generated_sequence_691: -6.500 kcal/mol
  - generated_sequence_267: -6.500 kcal/mol
  - generated_sequence_81: -6.500 kcal/mol
  - generated_sequence_731: -6.500 kcal/mol
  - generated_sequence_427:

8. ground-truth docking

In [10]:
!mk_prepare_receptor.py -i preprocessed/1Y6E_A_clean.pdb -o 1Y6E_A_clean -p -v --box_size 20 20 20 --box_center 18.148 21.074 -8.745
!vina --receptor 1Y6E_A_clean.pdbqt --ligand GSH.pdbqt --config 1Y6E_A_clean.box.txt --exhaustiveness=32 --out 1Y6E_A_clean.pdbqt

368.37s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



Files written:
  1Y6E_A_clean.pdbqt <-- static (i.e., rigid) receptor input file
1Y6E_A_clean.box.txt <-- Vina-style box dimension file
1Y6E_A_clean.box.pdb <-- PDB file to visualize the grid box


377.40s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, Journal of Computational Chemistry 31 (2010)  #
# 455-461                                                       #
#                                                               #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see http://vina.scripps.edu for more information.      #
#################################################################

Detected 104 CPUs
Reading input ... done.
Setting up the scoring function ... done.
Analyzing the binding site ... done.
Using random seed: -